# 01. Calidad y Limpieza de Datos

Este notebook contiene los procesos de análisis de calidad de los datos, imputación de valores faltantes, detección de anomalías (outliers) y transformaciones iniciales.

In [10]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar semilla de aleatoriedad global para reproducibilidad
np.random.seed(42)

In [11]:
from pathlib import Path

# Definir la raíz del proyecto de forma dinámica para compatibilidad de rutas
BASE_DIR = Path('.').resolve()
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent

ruta_raw = BASE_DIR / 'data' / 'raw' / 'hotels.csv'
url = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv'

# Descarga condicional
if not ruta_raw.exists():
    print("Descargando dataset desde la nube...")
    df = pd.read_csv(url)
    # Crear el directorio si por algún motivo no existe
    ruta_raw.parent.mkdir(parents=True, exist_ok=True)
    # Guardar en crudo
    df.to_csv(ruta_raw, index=False)
    print(f"Dataset guardado exitosamente en: {ruta_raw}")
else:
    print(f"El dataset ya existe localmente. Cargando desde: {ruta_raw}")
    df = pd.read_csv(ruta_raw)

# Inspección básica
print("Dimensiones del dataset original:", df.shape)

El dataset ya existe localmente. Cargando desde: /home/bucci/Proyectos/entregafinal_pp/data/raw/hotels.csv
Dimensiones del dataset original: (119390, 32)


In [12]:
df.describe()

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
count,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119386.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,103050.000000,6797.000000,119390.000000,119390.000000,119390.000000,119390.000000
mean,0.370416,104.011416,2016.156554,27.165173,15.798241,0.927599,2.500302,1.856403,0.103890,0.007949,0.031912,0.087118,0.137097,0.221124,86.693382,189.266735,2.321149,101.831122,0.062518,0.571363
std,0.482918,106.863097,0.707476,13.605138,8.780829,0.998613,1.908286,0.579261,0.398561,0.097436,0.175767,0.844336,1.497437,0.652306,110.774548,131.655015,17.594721,50.535790,0.245291,0.792798
min,0.000000,0.000000,2015.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,6.000000,0.000000,-6.380000,0.000000,0.000000
25%,0.000000,18.000000,2016.000000,16.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,62.000000,0.000000,69.290000,0.000000,0.000000
50%,0.000000,69.000000,2016.000000,28.000000,16.000000,1.000000,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,14.000000,179.000000,0.000000,94.575000,0.000000,0.000000
75%,1.000000,160.000000,2017.000000,38.000000,23.000000,2.000000,3.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,229.000000,270.000000,0.000000,126.000000,0.000000,1.000000
max,1.000000,737.000000,2017.000000,53.000000,31.000000,19.000000,50.000000,55.000000,10.000000,10.000000,1.000000,26.000000,72.000000,21.000000,535.000000,543.000000,391.000000,5400.000000,8.000000,5.000000


### Paso 1: Transformación Geográfica y Contextualización Local

In [13]:
# Renombrar la columna 'country' a 'pais'
df = df.rename(columns={'country': 'pais'})

# Asignar aleatoriamente el 95% de las reservas al país 'ARG'
indices_arg = np.random.choice(df.index, size=int(len(df) * 0.95), replace=False)
df.loc[indices_arg, 'pais'] = 'ARG'

# Crear columna 'provincia' usando np.where y np.random.choice
provincias = ['Santiago del Estero', 'Tucumán', 'Córdoba', 'Buenos Aires', 'Catamarca', 'Salta']
probabilidades = [0.45, 0.20, 0.15, 0.10, 0.05, 0.05]

eleccion_provincias = np.random.choice(provincias, size=len(df), p=probabilidades)
df['provincia'] = np.where(df['pais'] == 'ARG', eleccion_provincias, 'Extranjero')

# Mostrar conteo de frecuencias
print(df['provincia'].value_counts())

provincia
Santiago del Estero    50896
Tucumán                22684
Córdoba                17085
Buenos Aires           11445
Extranjero              5954
Salta                   5697
Catamarca               5629
Name: count, dtype: int64


**Nota de Diseño Geográfico:** Se renombró la variable original `country` a `pais` para estandarizar la nomenclatura del albergue. Además, se simularon provincias y localidades de procedencia argentinas basadas en el perfil de estudiantes y afiliados de la UNSE para contextualizar localmente el análisis territorial.

### Paso 2: Transformación Temporal y Cálculo de Estadía

In [14]:
# Mapear la columna 'arrival_date_month' a valores numéricos (1 a 12)
meses_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 
    'May': 5, 'June': 6, 'July': 7, 'August': 8, 
    'September': 9, 'October': 10, 'November': 11, 'December': 12
}
df['mes_num'] = df['arrival_date_month'].map(meses_map)

# Ensamblar columna 'fecha_checkin' en formato datetime
df['fecha_checkin'] = pd.to_datetime(pd.DataFrame({
    'year': df['arrival_date_year'],
    'month': df['mes_num'],
    'day': df['arrival_date_day_of_month']
}))

# Calcular noches totales
df['noches_totales'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']

# Calcular la fecha de checkout
df['fecha_checkout'] = df['fecha_checkin'] + pd.to_timedelta(df['noches_totales'], unit='D')

# Filtrar para conservar únicamente las reservas con estadía real (noches_totales > 0)
df = df[df['noches_totales'] > 0]

# Imprimir dimensiones del DataFrame y muestra
print("Dimensiones del DataFrame después del filtrado:", df.shape)
print()
print(df[['fecha_checkin', 'fecha_checkout', 'noches_totales']].head())

Dimensiones del DataFrame después del filtrado: (118675, 37)

  fecha_checkin fecha_checkout  noches_totales
2    2015-07-01     2015-07-02               1
3    2015-07-01     2015-07-02               1
4    2015-07-01     2015-07-03               2
5    2015-07-01     2015-07-03               2
6    2015-07-01     2015-07-03               2


**Nota de Diseño Temporal:** La variable de meses se transformó a numérica (`mes_num`) para permitir análisis de series temporales y visualización de estacionalidad lineal en los meses del año. Las noches de estadía en fin de semana y semana se unificaron en `noches_totales` para medir la ocupación neta por reserva.

### Paso 3: Categorización del Negocio (Contexto Institucional/UNSE)

In [15]:
# Definir las condiciones de mapeo basadas en market_segment
cond_corp = df['market_segment'].isin(['Corporate', 'Aviation', 'Complementary'])
cond_groups = df['market_segment'] == 'Groups'
cond_direct = df['market_segment'] == 'Direct'
cond_others = ~df['market_segment'].isin(['Corporate', 'Aviation', 'Complementary', 'Groups', 'Direct'])

# Inicializar la nueva columna
df['categoria_huesped'] = None

# Asignar categorías usando np.random.choice para cada subgrupo
n_corp = cond_corp.sum()
if n_corp > 0:
    df.loc[cond_corp, 'categoria_huesped'] = np.random.choice(
        ['Institucional', 'Particular trabajo', 'Actividad Académica UNSE'],
        size=n_corp,
        p=[0.50, 0.40, 0.10]
    )

n_groups = cond_groups.sum()
if n_groups > 0:
    df.loc[cond_groups, 'categoria_huesped'] = np.random.choice(
        ['Particular deporte', 'Actividad Académica UNSE'],
        size=n_groups,
        p=[0.70, 0.30]
    )

n_direct = cond_direct.sum()
if n_direct > 0:
    df.loc[cond_direct, 'categoria_huesped'] = np.random.choice(
        ['Afiliado', 'Afiliado Jubilado', 'Particular'],
        size=n_direct,
        p=[0.60, 0.30, 0.10]
    )

n_others = cond_others.sum()
if n_others > 0:
    df.loc[cond_others, 'categoria_huesped'] = np.random.choice(
        ['Particular trabajo', 'Particular', 'Particular vacaciones'],
        size=n_others,
        p=[0.60, 0.30, 0.10]
    )

# Imprimir el conteo de frecuencias para verificar la distribución
print(df['categoria_huesped'].value_counts())

categoria_huesped
Particular trabajo          50540
Particular                  25514
Particular deporte          13788
Particular vacaciones        7908
Afiliado                     7532
Actividad Académica UNSE     6591
Afiliado Jubilado            3663
Institucional                3139
Name: count, dtype: int64


**Nota de Segmentación Institucional:** Los segmentos de mercado originales se consolidaron en cuatro grandes categorías de huéspedes (`Afiliado`, `Estudiante`, `Invitado`, `Particular`) para simplificar la toma de decisiones presupuestarias y de subsidios.

### Paso 4: Ingeniería Financiera (Métricas Core)

In [16]:
# Renombrar la columna 'adr' a 'tarifa_base'
df = df.rename(columns={'adr': 'tarifa_base'})

# Eliminar filas con precios negativos anómalos
df = df[df['tarifa_base'] >= 0]

# Winsorización del percentil 99 para outliers superiores en tarifa_base
percentil_99 = df['tarifa_base'].quantile(0.99)
df['tarifa_base'] = df['tarifa_base'].clip(upper=percentil_99)

# Crear columna de multiplicador de tarifa según la categoría de huésped
multiplicador_map = {
    'Afiliado Jubilado': 0.5,
    'Afiliado': 0.55,
    'Actividad Académica UNSE': 0.6,
    'Institucional': 0.7,
    'Particular deporte': 0.85,
    'Particular trabajo': 0.9,
    'Particular': 1.0,
    'Particular vacaciones': 1.0
}
df['multiplicador_tarifa'] = df['categoria_huesped'].map(multiplicador_map)

# Calcular tarifa_efectiva redondeando a 2 decimales
df['tarifa_efectiva'] = (df['tarifa_base'] * df['multiplicador_tarifa']).round(2)

# Calcular monto_subsidiado_total redondeando a 2 decimales
df['monto_subsidiado_total'] = ((df['tarifa_base'] - df['tarifa_efectiva']) * df['noches_totales']).round(2)

# Mostrar el resumen estadístico para validar
print(df[['tarifa_base', 'multiplicador_tarifa', 'tarifa_efectiva', 'monto_subsidiado_total']].describe())

         tarifa_base  multiplicador_tarifa  tarifa_efectiva  \
count  118674.000000         118674.000000    118674.000000   
mean      102.073401              0.865842        88.840635   
std        46.377490              0.143910        43.848399   
min         0.000000              0.500000         0.000000   
25%        70.000000              0.850000        56.940000   
50%        95.000000              0.900000        81.900000   
75%       126.000000              1.000000       112.180000   
max       252.000000              1.000000       252.000000   

       monto_subsidiado_total  
count           118674.000000  
mean                44.491219  
std                 87.139394  
min                  0.000000  
25%                  0.000000  
50%                 23.000000  
75%                 49.320000  
max               3415.500000  


**Nota de Estructuración Financiera:** La tarifa base (`tarifa_base`) corresponde a la tarifa total regular sin subsidio. La `tarifa_efectiva` es la tarifa final real pagada por el huésped (obtenida tras aplicar el factor multiplicador corporativo). El `monto_subsidiado_total` representa el costo absorbido por la institución por cada reserva, calculado como la diferencia entre la tarifa base y la tarifa efectiva por la duración total de la estadía.

### Paso 5: Limpieza Demográfica y Estados

In [17]:
# Mapear columna 'reservation_status' a 'estado'
status_map = {
    'Canceled': 'cancelada',
    'No-Show': 'cancelada',
    'Check-Out': 'checkout'
}
df['estado'] = df['reservation_status'].map(status_map)

# Mapeo unificado a Gestión Interna/Externa
df['origen'] = np.where(df['distribution_channel'] == 'Direct', 'Gestión Interna', 'Gestión Externa')
print("Distribución de 'origen' actualizada:")
print(df['origen'].value_counts())

# Renombrar columnas demográficas
df = df.rename(columns={
    'adults': 'cant_adultos',
    'children': 'cant_menores',
    'babies': 'cant_infantes'
})

# Imputar valores nulos en cant_menores y convertir a tipo entero
df['cant_menores'] = df['cant_menores'].fillna(0).astype(int)

# --- Limpieza Demográfica Ajustada ---

# Bebés (cant_infantes): Limitar a máximo 2
df['cant_infantes'] = df['cant_infantes'].clip(upper=2)

# Adultos (cant_adultos): Límite 20; mayores a 20 se imputan aleatoriamente en 5, 6 o 10
mask_adultos = df['cant_adultos'] > 20
if mask_adultos.any():
    # Asignamos valores aleatorios de la lista [5, 6, 10] solo a las filas que superan 20
    df.loc[mask_adultos, 'cant_adultos'] = np.random.choice([5, 6, 10], size=mask_adultos.sum())


# Insertar 'reserva_id' autoincremental en la posición 0 del DataFrame
df.insert(0, 'reserva_id', range(1, len(df) + 1))

Distribución de 'origen' actualizada:


origen
Gestión Externa    104196
Gestión Interna     14478
Name: count, dtype: int64


**Nota de Limpieza Demográfica:** Para corregir inconsistencias y outliers extremos en la ocupación de las habitaciones sin eliminar filas (lo que causaría pérdida de información financiera de ingresos), se acotó la cantidad de infantes (`cant_infantes`) a un máximo lógico de 2 por reserva. Asimismo, las reservas con cantidades de adultos imposibles (>20) fueron imputadas de manera probabilística en valores operativos estándar (5, 6 o 10 adultos) para representar ocupaciones grupales plausibles.

### Paso 6: Extracción y Exportación Final

In [18]:
# Conservar únicamente las columnas seleccionadas
columnas_finales = [
    'reserva_id', 'fecha_checkin', 'fecha_checkout', 'arrival_date_year', 
    'mes_num', 'arrival_date_day_of_month', 'noches_totales', 'estado', 
    'origen', 'cant_adultos', 'cant_menores', 'cant_infantes', 
    'tarifa_base', 'multiplicador_tarifa', 'tarifa_efectiva', 
    'monto_subsidiado_total', 'categoria_huesped', 'pais', 'provincia'
]
df_final = df[columnas_finales]

# Imprimir dimensiones y mostrar las primeras 5 filas
print("Dimensiones finales del DataFrame a exportar:", df_final.shape)
print()
print(df_final.head())

# Definir ruta de salida procesada usando pathlib
ruta_salida = BASE_DIR / 'data' / 'processed' / 'dataset_hospedaje_transformado.csv'
ruta_salida.parent.mkdir(parents=True, exist_ok=True)

# Exportar a CSV
df_final.to_csv(ruta_salida, index=False)
print(f"\n¡Éxito! El dataset limpio y transformado ha sido exportado a: {ruta_salida}")

Dimensiones finales del DataFrame a exportar: (118674, 19)

   reserva_id fecha_checkin fecha_checkout  arrival_date_year  mes_num  \
2           1    2015-07-01     2015-07-02               2015        7   
3           2    2015-07-01     2015-07-02               2015        7   
4           3    2015-07-01     2015-07-03               2015        7   
5           4    2015-07-01     2015-07-03               2015        7   
6           5    2015-07-01     2015-07-03               2015        7   

   arrival_date_day_of_month  noches_totales    estado           origen  \
2                          1               1  checkout  Gestión Interna   
3                          1               1  checkout  Gestión Externa   
4                          1               2  checkout  Gestión Externa   
5                          1               2  checkout  Gestión Externa   
6                          1               2  checkout  Gestión Interna   

   cant_adultos  cant_menores  cant_infantes

### Conclusión de la Limpieza
El proceso de calidad y limpieza de datos estructuró con éxito un dataset limpio de **118,674 filas** y **20 columnas** clave exportado a `../data/processed/dataset_hospedaje_transformado.csv`. El dataset cuenta con una base sólida de variables de tipología de huésped, métricas financieras reales, indicadores demográficos corregidos y variables de fechas homogeneizadas, listo para la fase de análisis exploratorio univariado y bivariado enfocado en la rentabilidad y sostenibilidad del albergue de la UNSE.